# The company brain at 50,000 documents, with a large language model as its reasoning engine (A100)

**What this tests.** The company brain (v18) answers questions about one company from its documents by exact lookups:
the fact bank holds every field of every document, and a planner chains at most two steps. Round 4 tested it on 5,000
documents. This notebook tests it on **50,000**, the same 5,000 and 45,000 more of the benchmark's company
([EnterpriseRAG-Bench](https://github.com/onyx-dot-app/EnterpriseRAG-Bench), a simulated company with Slack, Gmail,
Linear, Jira, GitHub, HubSpot, Google Drive, Confluence and Fireflies records), and gives the brain a bigger reasoning
engine: **Qwen3.8-27B**, a 27-billion-parameter open model that thinks before it answers.

**Three ways to answer every question**

| arm | what answers | model |
|---|---|---|
| `v18` | the brain alone: exact lookups, plans of two steps, quoted sentences | none |
| `rag` | the model alone, reading the passages a keyword search finds (search, then a model: the usual AI search) | Qwen3.8-27B |
| `reasoner` | the model as the brain's reasoning engine: it reads the brain's answer, then can ask the brain more, search, read records, count and list with exact filters, up to 8 tool calls, before it answers | Qwen3.8-27B |

**The questions**
- Round 4's 587 questions, in the casual wordings that decided round 4 (and the generators' tidy wordings, reported), each
  kept only if it still has one right answer among 50,000 documents; a count whose filters now hold more documents is
  counted again.
- About 150 **complex questions** of two to four steps across systems: "how many Linear issues does the person assigned
  the issue linked from PR #1248 have?", "who owns the HubSpot account of the customer in the meeting X?", "in project P,
  who has the issue due first?", "how many open P1 issues does the assignee of ENG-12 have?".

What decides the result was fixed before any model answered: `docs/FACTBANK_50K_PLAN.md`. The score cell checks its rules.

**Before you start**
1. Runtime > Change runtime type > **A100 GPU** (80 GB runs the model in bf16; 40 GB runs Qwen's own 8-bit copy).
   High-RAM helps: the brain runs in several processes of about 3 GB each.
2. Secrets (the key icon): `HF_TOKEN` is optional (the model is not gated).
3. Run all. About 3.5 to 5 hours on an A100 80 GB: the benchmark clone, the draw, the bank and the questions about 40
   minutes, the brain's answers 20 to 30, plain search 30 to 45, the reasoning engine 2 to 3.5 hours (it thinks before
   every step). A 40 GB A100 takes about twice as long for the model arms: choose 80 GB if you can. `LIMIT = 40` gives a
   quick look at the first 40 questions in under an hour; its answers are kept, and a later run with `LIMIT = 0` goes on
   from them.

Results go to Google Drive (`MyDrive/cie_brain50k`) every few minutes while a step runs and when it ends; after a
disconnect, Run all goes on from there (the draw and the questions are taken from Drive, the bank is built again).

In [ ]:
#@title Parameters
import os
REPO = "https://github.com/Nikku03/integratedai"
BRANCH = "claude/epic-keller-gevn7j"
MODEL = "auto"                #@param ["auto", "qwen3.8-27b", "qwen3.8-27b-fp8"] {allow-input: true}
EFFORT = "medium"             #@param ["low", "medium", "xhigh"]
RUN_RAG = True                #@param {type:"boolean"}
RUN_REASONER = True           #@param {type:"boolean"}
MODEL_ON_TIDY = False         #@param {type:"boolean"}
LIMIT = 0                     #@param {type:"integer"}
WORKERS = 0                   #@param {type:"integer"}
SAVE_TO_DRIVE = True          #@param {type:"boolean"}
# MODEL: "auto" = Qwen3.8-27B in bf16 on an 80 GB GPU, Qwen's FP8 copy on a 40 GB one. Another name of
#   cie.factbank.reasoner.MODELS or a Hugging Face repository id works too (it is not what the plan decides on).
# EFFORT: Qwen3.8's reasoning effort (how long it thinks each turn). "medium" is the plan's setting.
# RUN_RAG / RUN_REASONER: the two model arms. The plan's rules 2 to 8 need both.
# MODEL_ON_TIDY: the model arms also answer the tidy wordings (reported, not decided; about 1.5 times the GPU time).
# LIMIT: 0 = every question. A number asks only the first questions of each folder: a quick look, not a decision.
# WORKERS: processes for the brain (each about 3 GB of memory). 0 picks by this machine's processors and memory.
# SAVE_TO_DRIVE: results go to MyDrive/cie_brain50k as each step finishes, so a disconnect loses little.
ON_COLAB = os.path.isdir("/content")
WORKDIR = "/content/integratedai/cie" if ON_COLAB else os.environ.get("CIE_COLAB_WORKDIR", os.getcwd())
BENCH = "/content/EnterpriseRAG-Bench-json" if ON_COLAB else os.environ.get("CIE_ERB_ROOT", os.path.join(os.getcwd(), "EnterpriseRAG-Bench"))
BENCH_COMMIT = "d36685e273713975ee20299bbf1ab64165575b3c"  # the benchmark's commit the plan was written with (2026-05-07)
# the 50,000-document folder stays on this machine's disk: the bank is read at random, which Drive is too slow for
WORK = "/content/brain50k/w50k" if ON_COLAB else os.environ.get("CIE_B50_WORK", os.path.join(os.getcwd(), "eval_out", "brain50k", "w50k"))
WORK_MESSY = WORK + "_messy"
INDEX = os.environ.get("B50_INDEX") or os.path.join(os.path.dirname(WORK), "index.json")
# local checks without a GPU: an OpenAI-compatible server (such as Ollama) stands in for vLLM
BACKEND = os.environ.get("B50_BACKEND", "vllm")
BASE_URL = os.environ.get("B50_BASE_URL", "")
MODEL = os.environ.get("B50_MODEL", MODEL)
LIMIT = int(os.environ.get("B50_LIMIT", LIMIT))
WORKERS = int(os.environ.get("B50_WORKERS", WORKERS))
EXPECT = {"dsids_sha256": "9a344aaac47f662c6c2158ba90899f670da2365a9d67747a578c5edbef0c67b5", "questions_sha256_messy": "a3ee9b75b6eaf94721b30d05e5b18b498f3d87cc67d71fc046e2fc91b4e0f4f8", "questions_sha256_tidy": "beeeac18480aa02223f0f44ae7d59016cf66a0d4c9fc6a0e6ecd8c52a2090dc2"}  # from the plan's own run of the draw and the questions (docs/FACTBANK_50K_PLAN.md)
print("model:", MODEL, "| effort:", EFFORT, "| arms: v18" + (", rag" if RUN_RAG else "") + (", reasoner" if RUN_REASONER else ""),
      "| questions:", LIMIT or "all", "| backend:", BACKEND, "| colab:", ON_COLAB)

In [ ]:
#@title GPU, memory and processors
import os, subprocess
q = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader,nounits", shell=True,
                   capture_output=True, text=True).stdout.strip().splitlines()
print(q[0] if q else "no GPU visible")
GPU_NAME, GPU_MEM_GB, DRIVER_MAJOR, CC = "", 0.0, 0, 0.0
if q:
    f = [x.strip() for x in q[0].split(",")]
    GPU_NAME, GPU_MEM_GB, DRIVER_MAJOR, CC = f[0], int(f[1]) / 1024, int(f[2].split(".")[0]), float(f[3])
if BACKEND == "vllm" and (RUN_RAG or RUN_REASONER):
    if not q:
        raise SystemExit("the model arms need a GPU: Runtime > Change runtime type > A100 GPU, then run all again")
    if CC < 8.0 or GPU_MEM_GB < 35:
        raise SystemExit(f"{GPU_NAME} ({GPU_MEM_GB:.0f} GB, compute capability {CC}) cannot hold the model: choose an A100 or H100")
if MODEL == "auto":
    MODEL = "qwen3.8-27b" if GPU_MEM_GB >= 70 else "qwen3.8-27b-fp8"
MAX_NUM_SEQS = 256 if GPU_MEM_GB >= 70 else 128
OLD_DRIVER = bool(q) and DRIVER_MAJOR < 580  # vLLM's default build needs driver 580 or newer
ram_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
cpus = os.cpu_count() or 2
if not WORKERS:
    WORKERS = max(2, min(10, cpus - 2, int((ram_gb - 25) // 4)))  # the model needs about 20 GB of memory while it loads
print(f"model {MODEL} | GPU memory {GPU_MEM_GB:.0f} GB | memory {ram_gb:.0f} GB | {cpus} processors -> {WORKERS} brain processes"
      + (f" | driver {DRIVER_MAJOR}: vLLM's CUDA 12.9 build" if OLD_DRIVER else ""))
if ram_gb < 40:
    print("note: little memory; the brain's processes may not fit. Runtime > Change runtime type > High-RAM helps.")

In [ ]:
#@title Clone the repository and install vLLM (a few minutes)
import importlib, os, subprocess, sys
if ON_COLAB and not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/integratedai"], check=True)
elif ON_COLAB:
    # an existing checkout moves to the latest commit of the branch: a re-run never tests stale code
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd="/content/integratedai", check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], cwd="/content/integratedai", check=True)
CODE = subprocess.run(["git", "log", "-1", "--format=%h %ci %s"], cwd=WORKDIR, capture_output=True, text=True).stdout.strip()
print("code at commit:", CODE)
if ON_COLAB and BACKEND == "vllm" and (RUN_RAG or RUN_REASONER):
    # vLLM brings its own torch; this kernel never imports torch, so no restart is needed
    if OLD_DRIVER:
        spec = ["vllm==0.30.0+cu129", "torch==2.13.0+cu129", "--extra-index-url", "https://wheels.vllm.ai/0.30.0/cu129",
                "--extra-index-url", "https://download.pytorch.org/whl/cu129"]
    else:
        spec = ["vllm==0.30.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec], check=True)
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", WORKDIR], check=True)  # without optional extras: no second torch
src = os.path.join(WORKDIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
os.chdir(WORKDIR)
if BACKEND == "vllm" and (RUN_RAG or RUN_REASONER):
    r = subprocess.run([sys.executable, "-c", "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, "
                        "'| GPU', torch.cuda.get_device_name(0))"], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-1500:])
    assert r.returncode == 0, "vLLM did not import (output above)"
import cie
print("cie", cie.__version__, "from", os.path.dirname(cie.__file__))

In [ ]:
#@title Keys from Colab secrets (never printed)
import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # not on Colab, or no such secret
        return os.environ.get(name)

if v := secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = v
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (not needed: the model is not gated)")

In [ ]:
#@title Results folder (Google Drive) and the steps' helpers
import hashlib, json, os, shutil, signal, subprocess, sys, threading, time, uuid
RESULTS = os.environ.get("B50_RESULTS") or os.path.join(os.path.dirname(WORK), "results")
if SAVE_TO_DRIVE and ON_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS = "/content/drive/MyDrive/cie_brain50k"
    except Exception as e:  # declined or unavailable
        print(f"Google Drive is not available ({type(e).__name__}): results stay on this runtime's disk and are lost with it")
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(os.path.dirname(WORK), exist_ok=True)
KEEP = ("haystack.json", "draw_report.json", "banks.json", "questions.jsonl", "questions_report.json", "set_aside.json", "arms50k.json",
        "factbank_v18.jsonl", "rag.jsonl", "reasoner.jsonl", "report50k.json", "report50k.md")
ANSWERS = ("factbank_v18.jsonl", "rag.jsonl", "reasoner.jsonl", "arms50k.json")
ROWS = ANSWERS[:3]  # answer files: Drive never gets a copy with fewer answers than it has
SAVE_EVERY = 300  # seconds: a step's answers so far go to Drive this often while it runs

def sha(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest() if os.path.exists(path) else None

def _lines(path):
    try:
        with open(path, "rb") as f:
            return sum(1 for _ in f)
    except OSError:
        return 0

def _copy(src, dst):
    # written whole (a temporary name, then renamed); a plain copy if the rename fails
    try:
        shutil.copy2(src, dst + ".tmp")
        os.replace(dst + ".tmp", dst)
    except OSError:
        shutil.copy2(src, dst)

def save():
    # every result file of both folders to RESULTS/<folder>/; what cannot be copied now (Drive away) is said, never raised
    for w in (WORK, WORK_MESSY):
        out = os.path.join(RESULTS, os.path.basename(w))
        try:
            os.makedirs(out, exist_ok=True)
        except OSError as e:
            print(f"(not saved to Drive now: {type(e).__name__}: {e})", flush=True)
            return
        for f in KEEP:
            src, dst = os.path.join(w, f), os.path.join(out, f)
            try:
                if not os.path.exists(src) or os.path.islink(src):  # the messy folder's links to the tidy one: saved there
                    continue
                if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src) and os.path.getmtime(dst) >= os.path.getmtime(src):
                    continue  # unchanged
                if f in ROWS and _lines(src) < _lines(dst):
                    continue
                _copy(src, dst)
            except OSError as e:
                print(f"(not saved to Drive now: {os.path.basename(w)}/{f}: {type(e).__name__}: {e})", flush=True)

def restore():
    # answers saved by an earlier runtime, put back when they answer the same questions (a row cut off half way by a
    # copy taken while it was written is dropped when the step reads the file, and asked again)
    for w in (WORK, WORK_MESSY):
        saved = os.path.join(RESULTS, os.path.basename(w))
        if sha(os.path.join(saved, "questions.jsonl")) is None or sha(os.path.join(saved, "questions.jsonl")) != sha(os.path.join(w, "questions.jsonl")):
            continue
        for f in ANSWERS:
            src, dst = os.path.join(saved, f), os.path.join(w, f)
            if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst)):
                shutil.copy2(src, dst)
                print("restored", os.path.basename(w), f)

def restore_draw():
    # the plan's draw saved by an earlier runtime, so a fresh runtime need not draw again
    saved = os.path.join(RESULTS, os.path.basename(WORK))
    if os.path.exists(os.path.join(WORK, "haystack.json")) or not os.path.exists(os.path.join(saved, "haystack.json")):
        return
    from cie.eval.brain50k import dsids_sha
    h = json.load(open(os.path.join(saved, "haystack.json")))
    if dsids_sha(h["dsids"]) != EXPECT.get("dsids_sha256") or h.get("root") != BENCH or not os.path.exists(os.path.join(saved, "draw_report.json")):
        return
    os.makedirs(WORK, exist_ok=True)
    for f in ("haystack.json", "draw_report.json"):
        shutil.copy2(os.path.join(saved, f), os.path.join(WORK, f))
    if not os.path.lexists(os.path.join(WORK, "index.json")):
        os.symlink(os.path.realpath(INDEX), os.path.join(WORK, "index.json"))
    print("restored the draw from", saved)

def restore_questions():
    # the plan's questions saved by an earlier runtime, so a fresh runtime need not draw them again
    pairs = ((WORK, "questions_sha256_tidy"), (WORK_MESSY, "questions_sha256_messy"))
    if all(os.path.exists(os.path.join(w, "questions.jsonl")) for w, _ in pairs):
        return
    if not all(sha(os.path.join(RESULTS, os.path.basename(w), "questions.jsonl")) == EXPECT.get(k) for w, k in pairs):
        return
    for w, _ in pairs:
        os.makedirs(w, exist_ok=True)
        for f in ("questions.jsonl", "questions_report.json", "set_aside.json"):
            shutil.copy2(os.path.join(RESULTS, os.path.basename(w), f), os.path.join(w, f))
    for f in ("haystack.json", "index.json", "factbank_v2.sqlite", "banks.json", "draw_report.json"):  # as the questions step links them
        if os.path.exists(os.path.join(WORK, f)) and not os.path.lexists(os.path.join(WORK_MESSY, f)):
            os.symlink(os.path.realpath(os.path.join(WORK, f)), os.path.join(WORK_MESSY, f))
    print("restored the questions from", RESULTS)

def env():
    return {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONHASHSEED": "0", "TOKENIZERS_PARALLELISM": "false", "PYTHONPATH": os.path.join(WORKDIR, "src")}

def _stop_step(marker):
    # every process a step started (its tool processes, vLLM's engine), found by the mark in its environment
    pids = []
    for d in os.listdir("/proc"):
        if d.isdigit() and int(d) != os.getpid():
            try:
                if f"B50_STEP={marker}".encode() in open(f"/proc/{d}/environ", "rb").read().split(b"\0"):
                    pids.append(int(d))
            except OSError:
                pass
    for sig in (signal.SIGTERM, signal.SIGKILL):
        for pid in pids:
            try:
                os.kill(pid, sig)
            except OSError:
                pass
        if pids:
            time.sleep(5)
    return pids

def run(args):
    # one step in its own process, its output shown as it comes; its answers so far go to Drive every SAVE_EVERY
    # seconds; processes it leaves behind (a crash that skips their clean-up) are stopped, so the cell always ends
    print("$", " ".join(args), flush=True)
    marker = uuid.uuid4().hex
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace",
                         env={**env(), "B50_STEP": marker}, cwd=WORKDIR)
    stop = threading.Event()

    def show():
        # the output is always read to its end: a full pipe would stop the step
        for line in p.stdout:
            try:
                print(line, end="", flush=True)
            except Exception:  # noqa: BLE001 - a line that cannot be shown is skipped
                pass

    def keep_saving():
        while not stop.wait(SAVE_EVERY):
            try:
                save()
            except Exception as e:  # Drive busy: the next try, or the save after the step
                print(f"(saving to Drive failed: {type(e).__name__}: {e})", flush=True)

    reader, saver = threading.Thread(target=show, daemon=True), threading.Thread(target=keep_saving, daemon=True)
    reader.start()
    saver.start()
    try:
        code = p.wait()
        reader.join(timeout=15)
        if reader.is_alive():
            print(f"(stopped {len(_stop_step(marker))} processes the step left running)", flush=True)
            reader.join(timeout=15)
    except BaseException:
        _stop_step(marker)
        raise
    finally:
        stop.set()
        saver.join()
        try:
            save()
        except Exception as e:
            print(f"(saving to Drive failed: {type(e).__name__}: {e})", flush=True)
    if code != 0:
        raise SystemExit(f"step failed with exit code {code} (output above)")

print("results:", RESULTS)
print("found from an earlier run:", ", ".join(sorted(os.listdir(RESULTS))) or "nothing (a fresh run)")

In [ ]:
#@title Get the benchmark: the repository's JSON records at the plan's commit (about 5 GB, a few minutes)
import os, shutil, subprocess
sources = os.path.join(BENCH, "generated_data", "sources")

def _complete():
    if not os.path.isfile(os.path.join(BENCH, "questions.jsonl")) or not os.path.isdir(os.path.join(BENCH, ".git")):
        return False
    listed = subprocess.run(["git", "ls-files", "generated_data/sources"], cwd=BENCH, capture_output=True, text=True)
    want = sum(1 for f in listed.stdout.splitlines() if f.endswith(".json"))
    have = sum(1 for _, _, fs in os.walk(sources) for f in fs if f.endswith(".json"))
    print(f"benchmark checkout: {have:,} of {want:,} JSON records on disk")
    return listed.returncode == 0 and want > 1000 and have == want

ok = _complete()
if not ok and os.path.isdir(os.path.join(BENCH, ".git")):
    subprocess.run(["git", "checkout", "-f", "HEAD", "--", "."], cwd=BENCH)  # restores files an interrupted checkout left out
    ok = _complete()
if not ok:
    shutil.rmtree(BENCH, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/onyx-dot-app/EnterpriseRAG-Bench", BENCH], check=True)
    ok = _complete()
assert ok, "the benchmark clone is incomplete; run this cell again"
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=BENCH, capture_output=True, text=True).stdout.strip()
if head != BENCH_COMMIT:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BENCH_COMMIT], cwd=BENCH, check=True)
    subprocess.run(["git", "checkout", "-q", BENCH_COMMIT], cwd=BENCH, check=True)
    head = BENCH_COMMIT
print("benchmark at commit", head[:12])

In [ ]:
#@title The corpus index: every document's id and file (reads all 512k records once, a few minutes; kept on Drive)
import json, os, shutil
from pathlib import Path
from cie.eval.bench_enterprise import build_index
saved = os.path.join(RESULTS, "index.json")
if not os.path.exists(INDEX) and os.path.exists(saved):
    shutil.copy2(saved, INDEX)
index = build_index(Path(BENCH) / "generated_data" / "sources", cache=Path(INDEX), log=print)
if ON_COLAB and not os.path.exists(saved):
    shutil.copy2(INDEX, saved)
print(f"{len(index):,} documents indexed")

In [ ]:
#@title Draw the 50,000 documents: round 4's 5,000 test documents and 45,000 more (a few minutes)
import json, os
from cie.eval.brain50k import dsids_sha
hay = os.path.join(WORK, "haystack.json")
restore_draw()
if not (os.path.exists(hay) and len(json.load(open(hay))["dsids"]) == 50000):
    run([sys.executable, "-u", "-m", "cie.eval.brain50k", "draw", "--index", INDEX, "--root", BENCH, "--out", WORK])
rep = json.load(open(os.path.join(WORK, "draw_report.json")))
got = dsids_sha(json.load(open(hay))["dsids"])
print(f"{rep['documents']:,} documents: {rep['base']:,} of round 4 and {rep['added']:,} more; left out: {rep['skipped']}")
print("by system:", rep["by_source"])
print("the same documents as the plan's draw:", got == EXPECT.get("dsids_sha256"), f"({got[:16]})")

In [ ]:
#@title Build the fact bank of the 50,000 documents (a few minutes)
import json, os
from pathlib import Path
from cie.eval import brain_test as H
banks = os.path.join(WORK, "banks.json")
fresh = os.path.exists(banks) and json.load(open(banks)).get("factbank_v2", {}).get("haystack_sha256") == H.haystack_sha(Path(WORK))
if not (fresh and os.path.exists(os.path.join(WORK, "factbank_v2.sqlite"))):
    run([sys.executable, "-u", "-m", "cie.eval.brain_test", "build", "--work", WORK])
b = json.load(open(banks))["factbank_v2"]
print(f"bank: {b['sources']:,} documents, {b['facts']:,} facts, {b['bytes'] / 1e9:.2f} GB, built in {b['seconds']:.0f} s, check {b['check_ok']}")

In [ ]:
#@title The questions: round 4's that still hold among 50,000 documents, and the complex ones (10 to 20 minutes)
import json, os
restore_questions()
if not (os.path.exists(os.path.join(WORK, "questions.jsonl")) and os.path.exists(os.path.join(WORK_MESSY, "questions.jsonl"))):
    run([sys.executable, "-u", "-m", "cie.eval.brain50k", "questions", "--work", WORK])
for w, key in ((WORK_MESSY, "questions_sha256_messy"), (WORK, "questions_sha256_tidy")):
    r = json.load(open(os.path.join(w, "questions_report.json")))
    print(f"{os.path.basename(w)}: {r['questions']} questions {r['by_family']}")
    print("  the same questions as the plan's:", sha(os.path.join(w, "questions.jsonl")) == EXPECT.get(key))
r4 = r["round4"]
print(f"round 4's questions: {r4['kept']} of {r4['questions']} still hold, {r4['recounted']} of them counted again; set aside: {r4['set_aside_reasons']}")
print("complex questions by kind:", r["complex"]["drawn"])
restore()
save()

In [ ]:
#@title The brain alone (v18): every question of both folders, in parallel processes (about 20 to 30 minutes)
for w in (WORK_MESSY, WORK):  # the messy folder first: it decides; each folder saved when it is done
    run([sys.executable, "-u", "-m", "cie.eval.brain50k", "ask", "--arm", "v18", "--work", w, "--workers", str(WORKERS)]
        + (["--limit", str(LIMIT)] if LIMIT else []))
    save()

In [ ]:
#@title The model arms: plain search (rag) and the reasoning engine (reasoner), each in its own process
works = [WORK_MESSY] + ([WORK] if MODEL_ON_TIDY else [])
common = ["--work", *works, "--model", MODEL, "--effort", EFFORT, "--workers", str(WORKERS), "--max-num-seqs", str(MAX_NUM_SEQS)]
if LIMIT:
    common += ["--limit", str(LIMIT)]
if BACKEND != "vllm":
    common += ["--base-url", BASE_URL]
for arm, on in (("rag", RUN_RAG), ("reasoner", RUN_REASONER)):
    if on:
        run([sys.executable, "-u", "-m", "cie.eval.brain50k", "ask", "--arm", arm, *common])
        save()

In [ ]:
#@title Score: every arm by family, the complex questions by kind, and the plan's rules (docs/FACTBANK_50K_PLAN.md)
import json, os
from IPython.display import Markdown, display
run([sys.executable, "-u", "-m", "cie.eval.brain50k", "score", "--messy", WORK_MESSY, "--tidy", WORK])
save()
display(Markdown(open(os.path.join(WORK_MESSY, "report50k.md")).read()))
if LIMIT:
    print(f"LIMIT = {LIMIT}: a quick look at the first questions, not a decision")

In [ ]:
#@title Look at a few complex questions: each arm's answer and the reasoning engine's steps
import json, os, random
qs = {json.loads(l)["id"]: json.loads(l) for l in open(os.path.join(WORK_MESSY, "questions.jsonl"))}
ans = {a: {json.loads(l)["id"]: json.loads(l) for l in open(os.path.join(WORK_MESSY, f))}
       for a, f in (("v18", "factbank_v18.jsonl"), ("rag", "rag.jsonl"), ("reasoner", "reasoner.jsonl")) if os.path.exists(os.path.join(WORK_MESSY, f))}
cx = [i for i in qs if qs[i]["family"] == "complex" and all(i in a for a in ans.values())]
for i in random.Random(2).sample(cx, min(4, len(cx))):
    q = qs[i]
    print("=" * 100, f"\n{q['kind']} ({q['steps']} steps): {q['question']}\nexpected: {q['expected']['value']}")
    for a, rows in ans.items():
        print(f"  {a:9s} {str(rows[i]['answer'])[:200]}")
    for s in ans.get("reasoner", {}).get(i, {}).get("steps", []):
        print("     ", json.dumps({k: v for k, v in s.items() if k in ("tool", "args", "answer", "gave_up", "error")})[:220])

In [ ]:
#@title Where the results are
import os
print(f"Results: {RESULTS}" + (" (Google Drive)" if RESULTS.startswith("/content/drive") else " (this runtime's disk)"))
for root, _, files in os.walk(RESULTS):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, RESULTS):45s} {os.path.getsize(p) / 1e6:8.2f} MB")
print("Paste the report above (or report50k.md) into the chat to have it written up.")